In [1]:
from preprocessing import dataset_preprocessing 

from preprocessing import detect_categories 

from tree import DecisionTree 

from sklearn.model_selection import train_test_split

import pandas as pd

import numpy as np

from itertools import product

def training_split(dataset) : 

    train_data , test_data = train_test_split(dataset , test_size = 0.2 , random_state = 42 , stratify = dataset['satisfaction'])     
    return train_data , test_data

def tuning_split(train_data) :
    
    tuning_train , tuning_validation = train_test_split(train_data , test_size = 0.2 , random_state = 42 , stratify = train_data['satisfaction'])
    return tuning_train , tuning_validation 


def coarse_search(tuning_train , feats_dict , tuning_validation , n_iter , sampling_ratio):

    coarse_search_df = {"max_depth" : [] , "min_samples_split" : [] , "criterion" : [] ,
                         "train_F1-Score" : [] , "train_Acc" : [] , "validation_F1-Score" : [] , "validation_Acc" : []}
    _ , coarse_sample = train_test_split(tuning_train , test_size = sampling_ratio , random_state = 42 , stratify = tuning_train['satisfaction'])

    search_domain = { "max_depth" : [3,5,7,9] , "min_samples_split" : [2,10,20,50] ,"criterion" : ['gini','gain'] }
    hyper_params = {}

    for _ in range(n_iter) :
        
        for hyper_param in list(coarse_search_df.keys())[:-4]:
           hyper_params[hyper_param] = np.random.choice(search_domain[hyper_param])
           coarse_search_df[hyper_param].append(hyper_params[hyper_param])
        
        decision_tree = DecisionTree()
        decision_tree.training(coarse_sample , hyper_params ,feats_dict.copy())
        
        F1_Score , Acc = decision_tree.tree_evaluation(coarse_sample)
        coarse_search_df["train_F1-Score"].append(F1_Score)
        coarse_search_df["train_Acc"].append(Acc)
        
        F1_Score , Acc = decision_tree.tree_evaluation(tuning_validation)
        coarse_search_df["validation_F1-Score"].append(F1_Score)
        coarse_search_df["validation_Acc"].append(Acc)

        #  pruning 

    return pd.DataFrame(coarse_search_df)
 
def domain_generation(coarse_search_df , row , hyper_param , tolerance):
    
    value = coarse_search_df.loc[row , hyper_param ]
    if value is None :
        return [None]
    elif value in ['gini','gain'] :
        return [value]
    else :
        value = int(value)
        return [x for x in range(value - tolerance , value + tolerance + 1) if x >= 1]


def fine_search(tuning_train , feats_dict , tuning_validation , coarse_search_df ,  top_k) :

    fine_search_df = {"max_depth" : [] , "min_samples_split" : [] , "criterion" : [] , "train_F1-Score" : [] , 
                      "train_Acc" : [] , "validation_F1-Score" : [] ,"validation_Acc" : []}
    
    coarse_search_df["weighted_score"] = 0.7 * coarse_search_df["validation_Acc"] + 0.3 * coarse_search_df["validation_F1-Score"]
    coarse_search_df.to_csv("../data/coarse_search.csv" , index = False)
    coarse_search_df = coarse_search_df.sort_values(by = "weighted_score" , ascending = False ).reset_index(drop = True)
    hyper_params = {}

    for i in range(top_k):
        
        grid_search = {}
        tolerance = [1,0,0]
        for index , hyper_param in enumerate(list(fine_search_df.keys())[:-4]) :
            grid_search[hyper_param] = tolerance[index]
        h_t = list(grid_search.items())
        for hyper_param , tolerance in  h_t:
            grid_search[hyper_param] = domain_generation(coarse_search_df , i , hyper_param , tolerance)
        
        for combination in product(grid_search["max_depth"],grid_search["min_samples_split"],grid_search["criterion"]) :
            for index , hyper_param in enumerate(grid_search.keys()):
                hyper_params[hyper_param] = combination[index]
                fine_search_df[hyper_param].append(combination[index])    
            
            decision_tree = DecisionTree()
            decision_tree.training(tuning_train , hyper_params , feats_dict.copy())
            F1_Score , Acc = decision_tree.tree_evaluation(tuning_train)
            fine_search_df["train_F1-Score"].append(F1_Score)
            fine_search_df["train_Acc"].append(Acc)

            F1_Score , Acc = decision_tree.tree_evaluation(tuning_validation)
            fine_search_df["validation_F1-Score"].append(F1_Score)
            fine_search_df["validation_Acc"].append(Acc)
            
            # pruning 

    fine_search_df = pd.DataFrame(fine_search_df)
    fine_search_df["weighted_score"] = 0.7 * fine_search_df["validation_Acc"] + 0.3 * fine_search_df["validation_F1-Score"]
    optimal_row = fine_search_df["weighted_score"].idxmax()
    fine_search_df.to_csv("../data/fine_search.csv" , index = False )

    for hyper_param in hyper_params.keys():
        hyper_params[hyper_param] = fine_search_df.loc[optimal_row , hyper_param]
    
    return hyper_params
    
def hyperparameter_tuning(train_data , feats_dict , n_iter , sampling_ratio , top_k):

    del feats_dict['satisfaction']
    tuning_train , tuning_validation = tuning_split(train_data)
    coarse_search_df = coarse_search(tuning_train , feats_dict , tuning_validation , n_iter , sampling_ratio)
    return fine_search(tuning_train , feats_dict , tuning_validation , coarse_search_df , top_k)

def training(train_data , test_data , optimal_hyper_params , feats_dict):

    del feats_dict['satisfaction']
    decision_tree = DecisionTree()
    decision_tree.training(train_data , optimal_hyper_params , feats_dict.copy())
    train_F1_score  , train_acc = decision_tree.tree_evaluation(train_data)
    test_F1_Score , test_acc = decision_tree.tree_evaluation(test_data)
    
    # pruning 

    return decision_tree , (train_F1_score , train_acc) , (test_F1_Score , test_acc)


In [ ]:
# Decision_tree 1 trained on original dataset 

processed_dataset , original_column_categories = dataset_preprocessing(["id"],["Arrival Delay in Minutes"],["Arrival Delay in Minutes","Departure Delay in Minutes"],
{"Flight Distance" : 4 , "Arrival Delay in Minutes" : 4 , "Departure Delay in Minutes" : 4},{"Age" : 4},{})

# updated_columns_categories  

feats_dict = detect_categories(processed_dataset)

train_data , test_data = training_split(processed_dataset)

optimal_hyper_params = hyperparameter_tuning(train_data ,feats_dict.copy() , 40 , 0.3 , 3)

decision_tree , train_metrics , test_metrics = training(train_data , test_data , optimal_hyper_params , feats_dict.copy())

print(f"train F1 score / train accuracy : {train_metrics}")

print(f"test F1 score / test accuracy : {test_metrics}")

train F1 score / train accuracy : (np.float64(0.38560682435052346), np.float64(0.40750454146265175))
test F1 score / test accuracy : (np.float64(0.14983474109438122), np.float64(0.10466291323805399))


In [3]:
import graphviz

from IPython.display import Image

dot = graphviz.Digraph(comment = 'Decision_tree 1')

decision_tree.tree_visualization(decision_tree.root , dot , original_column_categories)

dot.render(filename = 'Decision_tree_1' , format = 'png' , view = False )

Image(filename = "Decision_tree_1.png")

dot: graph is too large for cairo-renderer bitmaps. Scaling by 0.0485885 to fit


In [ ]:
# Decision_tree 2 trained on feature engineered dataset 

processed_dataset , original_column_categories = dataset_preprocessing(["id","Gender","Arrival Delay in Minutes","Departure Delay in Minutes","Departure/Arrival time convenient","Gate location"],[],
[],{"Flight Distance" : 4 },{"Age" : 4},{("Inflight wifi service","Online boarding") : ("PCA","Online services") , 
("Seat comfort","Inflight entertainment","Cleanliness") : ("Mean","Comfort amenities") })

# updated_columns_categories  

feats_dict = detect_categories(processed_dataset)

train_data , test_data = training_split(processed_dataset)

optimal_hyper_params = hyperparameter_tuning(train_data ,feats_dict.copy() , 40 , 0.3 , 3) 

decision_tree , train_metrics , test_metrics = training(train_data , test_data , optimal_hyper_params , feats_dict.copy())

print(f"train F1 score / train accuracy : {train_metrics}")

print(f"test F1 score / test accuracy : {test_metrics}")

In [ ]:
dot = graphviz.Digraph(comment = 'Decision_tree 2')

decision_tree.tree_visualization(decision_tree.root , dot , original_column_categories)

dot.render(filename = 'Decision_tree_2' , format = 'png' , view = False )

Image(filename = "Decision_tree_2.png")